In [2]:
"""
FINN TFC-W1A1 Model with modified hidden layer widths: 784 -> 256 -> 256 -> 10 BNN for MNIST
Progressive bit-width training method
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import brevitas.nn as qnn
from brevitas.quant import Int8WeightPerTensorFloat, Int8ActPerTensorFloat, SignedBinaryWeightPerTensorConst
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time
import os
import matplotlib.pyplot as plt
import json

print("Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

Libraries imported successfully
PyTorch version: 2.10.0+cu130
Device available: CUDA


In [6]:
class TFC_Progressive(nn.Module):
    """
    Class that describes the layer architecture.
    From 784 to 256  to 256  to 10 neurons
    """
    def __init__(self, weight_bit_width=8, act_bit_width=8):
        super(TFC_Progressive, self).__init__()
        
        self.weight_bit_width = weight_bit_width
        self.act_bit_width = act_bit_width
        
        # Layer 1: 784 to 256 
        self.fc1 = qnn.QuantLinear(
            in_features=784,
            out_features=256 ,
            bias=True,
            weight_bit_width=weight_bit_width,
            weight_quant=SignedBinaryWeightPerTensorConst  
        )
        self.bn1 = nn.BatchNorm1d(256 )
        
        if act_bit_width == 1:
            # For 1-bit, use standard ReLU (no quantization)
            self.act1 = nn.ReLU()
        else:
            # For 2-bit and higher, use QuantReLU
            self.act1 = qnn.QuantReLU(bit_width=act_bit_width)
        
        # Layer 2: 256  to 256 
        self.fc2 = qnn.QuantLinear(
            in_features=256 ,
            out_features=256 ,
            bias=True,
            weight_bit_width=weight_bit_width,
            weight_quant=SignedBinaryWeightPerTensorConst  
        )
        self.bn2 = nn.BatchNorm1d(256 )
        
        if act_bit_width == 1:
            self.act2 = nn.ReLU()
        else:
            self.act2 = qnn.QuantReLU(bit_width=act_bit_width)
        
        # Layer 3: 256  to 10
        self.fc3 = qnn.QuantLinear(
            in_features=256 ,
            out_features=10,
            bias=True,
            weight_bit_width=weight_bit_width,
            weight_quant=SignedBinaryWeightPerTensorConst  
        )
    
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.act1(self.bn1(self.fc1(x)))
        x = self.act2(self.bn2(self.fc2(x)))
        x = self.fc3(x)
        return x

# quantLinear is memory and hardware efficient, since we use XNOR (LUTs) instead of DSPs like traditional NNs

In [7]:
def transfer_weights(source_model, target_model):
    """
    Transfer weights between models with different quantization
    since we are going from 8-bit to 4-bit to 2-bit to 1-bit
    Only transfer the actual weight values.
    Args:
        source_model: Trained model
        target_model: New model to initialize

    Returns:
        target_model: New model with transferred weights
    
    References:
        https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html
    """
    source_dict = source_model.state_dict()
    target_dict = target_model.state_dict()

    transfer_model_keys = [
        'fc1.weight', 'fc1.bias',
        'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked',
        'fc2.weight', 'fc2.bias',
        'bn2.weight', 'bn2.bias', 'bn2.running_mean', 'bn2.running_var', 'bn2.num_batches_tracked',
        'fc3.weight', 'fc3.bias'
    ]

    transferred = 0
    for key in transfer_model_keys:
        if key in source_dict and key in target_dict:
            target_dict[key] = source_dict[key].clone() # PyTorch function that creates independent copy
            transferred += 1

    # Load and confirm model was successfully transferred.
    target_model.load_state_dict(target_dict, strict=False)
    print(f"Transferred {transferred} parameter tensors")

    return target_model


In [8]:
def train_stage(model, train_loader, test_loader, optimizer, criterion, 
                    device, num_epochs, stage_name):
    """
    Train model for one quantization stage
    
    Args:
        model: The model to train
        train_loader: DataLoader for training data
        test_loader: DataLoader for test data
        optimizer: PyTorch optimizer (e.g., Adam)
        criterion: Loss function (e.g., CrossEntropyLoss)
        device: 'cpu' or 'cuda'
        num_epochs: How many epochs to train
        stage_name: String like "STAGE 1: 8-bit Quantization"
    
    Returns:
        best_acc: Best test accuracy achieved (float)
        history: Dictionary with training metrics (dict)
    
    References:
        Training loop: https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
        Evaluation: https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html#test-the-network-on-the-test-data
    """
    
    print(f"\n{'='*70}")
    print(f"{stage_name}")
    print(f"{'='*70}")
    
    best_acc = 0
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

    # Training loop
    for epoch in range(num_epochs):
        model.train() # Set model to training mode
        
        # Initialize epoch metrics
        running_loss = 0.0
        correct = 0
        total = 0

        # Loop though train_loader batches
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Check for NaN
            if torch.isnan(loss):
                print(f"  NaN loss detected at epoch {epoch+1}, batch {batch_idx}")
                print(f"  Skipping this batch...")
                continue
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            if batch_idx % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}] Batch [{batch_idx}/{len(train_loader)}] '
                      f'Loss: {loss.item():.4f}')
        
        train_loss = running_loss / len(train_loader)
        train_acc = 100. * correct / total if total > 0 else 0.0
        
        # Testing
        model.eval()
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                test_total += labels.size(0)
                test_correct += predicted.eq(labels).sum().item()
        
        test_acc = 100. * test_correct / test_total
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        
        print(f'Epoch [{epoch+1}/{num_epochs}] '
              f'Train Loss: {train_loss:.4f} | '
              f'Train Acc: {train_acc:.2f}% | '
              f'Test Acc: {test_acc:.2f}%')
        
        if test_acc > best_acc:
            best_acc = test_acc
    
    print(f"\n{stage_name} Complete - Best Test Acc: {best_acc:.2f}%")
    
    return best_acc, history

In [9]:
def train_progressive_bnn():
    """
    Progressive training pipeline: 8-bit -> 4-bit -> 2-bit -> 1-bit
    
    Returns:
        model_1bit: Final trained 1-bit model
        history_dict: Dictionary containing training history for all stages
    
    References:
        Data loading: https://pytorch.org/tutorials/beginner/basics/data_tutorial.html
        MNIST dataset: https://pytorch.org/vision/stable/datasets.html#mnist
        Progressive training: Transfer learning concept
    """
    
    # Data loading
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    
    train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST('./data', train=False, transform=transform)
    
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    device = torch.device('cpu')
    criterion = nn.CrossEntropyLoss()
    
    start_time = time.time()
    
    print("="*70)
    print("PROGRESSIVE BNN TRAINING - COMPLETE 4-STAGE VERSION")
    print("="*70)
    print("Architecture: 784 → 256  → 256  → 10")
    print("Stages: 8-bit → 4-bit → 2-bit → 1-bit")
    print("="*70)
    

    # STAGE 1: 8-bit Training (20 epochs)

    print("\nStarting Stage 1: 8-bit quantization")
    model_8bit = TFC_Progressive(weight_bit_width=8, act_bit_width=8).to(device)
    optimizer = optim.Adam(model_8bit.parameters(), lr=0.001)
    
    best_acc_8bit, hist_8bit = train_stage(
        model_8bit, train_loader, test_loader, optimizer, criterion, device,
        num_epochs=20, stage_name="STAGE 1: 8-bit Quantization"
    )
    
    torch.save(model_8bit.state_dict(), 'model_8bit.pth')
    print(f"Saved model_8bit.pth")
    

    # STAGE 2: 4-bit Fine-tuning (10 epochs)

    print("\nStarting Stage 2: 4-bit quantization")
    model_4bit = TFC_Progressive(weight_bit_width=4, act_bit_width=4).to(device)
    print("Transferring weights from 8-bit to 4-bit model...")
    model_4bit = transfer_weights(model_8bit, model_4bit)
    
    optimizer = optim.Adam(model_4bit.parameters(), lr=0.0001)
    
    best_acc_4bit, hist_4bit = train_stage(
        model_4bit, train_loader, test_loader, optimizer, criterion, device,
        num_epochs=10, stage_name="STAGE 2: 4-bit Quantization"
    )
    
    torch.save(model_4bit.state_dict(), 'model_4bit.pth')
    print(f"Saved model_4bit.pth")
    

    # STAGE 3: 2-bit Fine-tuning (10 epochs)

    print("\nStarting Stage 3: 2-bit quantization")
    model_2bit = TFC_Progressive(weight_bit_width=2, act_bit_width=2).to(device)
    print("Transferring weights from 4-bit to 2-bit model...")
    model_2bit = transfer_weights(model_4bit, model_2bit)
    
    # Use even lower learning rate for 2-bit
    optimizer = optim.Adam(model_2bit.parameters(), lr=0.00005)
    
    best_acc_2bit, hist_2bit = train_stage(
        model_2bit, train_loader, test_loader, optimizer, criterion, device,
        num_epochs=10, stage_name="STAGE 3: 2-bit Quantization"
    )
    
    torch.save(model_2bit.state_dict(), 'model_2bit.pth')
    print(f"Saved model_2bit.pth")
    

    # STAGE 4: 1-bit Fine-tuning

    print("\nStarting Stage 4: 1-bit quantization (FINAL BNN)")
    model_1bit = TFC_Progressive(weight_bit_width=1, act_bit_width=1).to(device)
    print("Transferring weights from 2-bit to 1-bit model...")
    model_1bit = transfer_weights(model_2bit, model_1bit)
    
    # Very low learning rate for 1-bit
    optimizer = optim.Adam(model_1bit.parameters(), lr=0.00001)
    
    best_acc_1bit, hist_1bit = train_stage(
        model_1bit, train_loader, test_loader, optimizer, criterion, device,
        num_epochs=15, stage_name="STAGE 4: 1-bit Quantization (FINAL BNN)"
    )
    
    torch.save(model_1bit.state_dict(), 'tfc_w1a1_final.pth')
    print(f"Saved tfc_w1a1_final.pth")
    

    # Final Summary

    total_time = time.time() - start_time
    
    print("\n" + "="*70)
    print("TRAINING COMPLETE")
    print("="*70)
    print(f"Total training time: {total_time/60:.1f} minutes")
    print(f"\nAccuracy Summary:")
    print(f"  Stage 1 (8-bit):  {best_acc_8bit:.2f}%")
    print(f"  Stage 2 (4-bit):  {best_acc_4bit:.2f}%")
    print(f"  Stage 3 (2-bit):  {best_acc_2bit:.2f}%")
    print(f"  Stage 4 (1-bit):  {best_acc_1bit:.2f}% ← FINAL BNN")
    print("="*70)
    
    # Accuracy degradation analysis
    degradation_4bit = best_acc_8bit - best_acc_4bit
    degradation_2bit = best_acc_4bit - best_acc_2bit
    degradation_1bit = best_acc_2bit - best_acc_1bit
    total_degradation = best_acc_8bit - best_acc_1bit
    
    print(f"\nAccuracy Degradation Analysis:")
    print(f"  8-bit → 4-bit:  {degradation_4bit:+.2f}%")
    print(f"  4-bit → 2-bit:  {degradation_2bit:+.2f}%")
    print(f"  2-bit → 1-bit:  {degradation_1bit:+.2f}%")
    print(f"  Total (8→1):    {total_degradation:+.2f}%")
    print("="*70)
    
    return model_1bit, {
        '8bit': hist_8bit,
        '4bit': hist_4bit,
        '2bit': hist_2bit,
        '1bit': hist_1bit
    }


In [12]:
# Run progressive training
print("Starting Progressive BNN Training...")
print("This will take approximately 20-30 minutes on CPU")
print("Expected final accuracy: 93-95%\n")

final_model = train_progressive_bnn()

Starting Progressive BNN Training...
This will take approximately 20-30 minutes on CPU
Expected final accuracy: 93-95%

PROGRESSIVE BNN TRAINING - COMPLETE 4-STAGE VERSION
Architecture: 784 → 256  → 256  → 10
Stages: 8-bit → 4-bit → 2-bit → 1-bit

Starting Stage 1: 8-bit quantization

STAGE 1: 8-bit Quantization
Epoch [1/20] Batch [0/469] Loss: 2.9806


KeyboardInterrupt: 

In [10]:
# Load 1-bit model
device = torch.device('cpu')
model_final = TFC_Progressive(weight_bit_width=1, act_bit_width=1).to(device)

# Load trained weights
checkpoint = torch.load('tfc_w1a1_final.pth')
model_final.load_state_dict(checkpoint)
model_final.eval()

print("Model loaded")

Model loaded


In [11]:
def export_binary_weights(model, output_dir='./fpga_weights'):
    """
    Extract binary weights from trained model for FPGA implementation
    
    Args:
        model: Trained TFC_Progressive model
        output_dir: Directory to save weight files
    
    Returns:
        Dictionary of binary weight matrices
    
    Reference:
        Binary weight extraction for hardware deployment
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    model.eval()
    weights_dict = {}
    
    # Extract fc1 weights
    fc1_weights_float = model.fc1.weight.data.cpu().numpy()
    fc1_binary = np.sign(fc1_weights_float)
    
    # Save as .npy file
    np.save(f'{output_dir}/fc1_weights.npy', fc1_binary)
    
    # Save as .txt for HLS (human-readable)
    np.savetxt(f'{output_dir}/fc1_weights.txt', fc1_binary, fmt='%d')
    
    # Print statistics
    print(f"FC1: {fc1_binary.shape}, unique values: {np.unique(fc1_binary)}")
    
    # Extract fc2 weights
    fc2_weights_float = model.fc2.weight.data.cpu().numpy()
    fc2_binary = np.sign(fc2_weights_float)
    
    # Save as .npy file
    np.save(f'{output_dir}/fc2_weights.npy', fc2_binary)
    
    # Save as .txt for HLS (human-readable)
    np.savetxt(f'{output_dir}/fc2_weights.txt', fc2_binary, fmt='%d')
    
    # Print statistics
    print(f"FC2: {fc2_binary.shape}, unique values: {np.unique(fc2_binary)}")

    # Extract fc3 weights
    fc3_weights_float = model.fc3.weight.data.cpu().numpy()
    fc3_binary = np.sign(fc3_weights_float)
    
    # Save as .npy file
    np.save(f'{output_dir}/fc3_weights.npy', fc3_binary)
    
    # Save as .txt for HLS (human-readable)
    np.savetxt(f'{output_dir}/fc3_weights.txt', fc3_binary, fmt='%d')
    
    # Print statistics
    print(f"FC3: {fc3_binary.shape}, unique values: {np.unique(fc3_binary)}")
    
    weights_dict = {
    'fc1': fc1_binary,
    'fc2': fc2_binary,
    'fc3': fc3_binary
    }
    return weights_dict

# Run export
weights = export_binary_weights(model_final)

FC1: (256, 784), unique values: [-1.  1.]
FC2: (256, 256), unique values: [-1.  1.]
FC3: (10, 256), unique values: [-1.  1.]


In [12]:
def export_batchnorm_params(model, output_dir='./fpga_weights'):
    """
    Extract BatchNorm parameters for FPGA implementation
    
    BatchNorm formula: y = gamma * (x - mean) / sqrt(var + eps) + beta
    
    For FPGA, fold into: y = scale * x + bias
    where: scale = gamma / sqrt(var + eps)
           bias = beta - (gamma * mean) / sqrt(var + eps)
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Extract bn1 parameters
    gamma1 = model.bn1.weight.data.cpu().numpy()
    beta1 = model.bn1.bias.data.cpu().numpy()
    mean1 = model.bn1.running_mean.cpu().numpy()
    var1 = model.bn1.running_var.cpu().numpy()
    eps1 = model.bn1.eps
    
    # Compute folded parameters
    scale1 = gamma1 / np.sqrt(var1 + eps1)
    bias1 = beta1 - (gamma1 * mean1) / np.sqrt(var1 + eps1)

    # Save parameters
    np.save(f'{output_dir}/bn1_scale.npy', scale1)
    np.save(f'{output_dir}/bn1_bias.npy', bias1)
    np.savetxt(f'{output_dir}/bn1_scale.txt', scale1, fmt='%.6f')
    np.savetxt(f'{output_dir}/bn1_bias.txt', bias1, fmt='%.6f')

    print(f"BN1: scale shape={scale1.shape}, bias shape={bias1.shape}")

    # Extract bn2 parameters
    gamma2 = model.bn2.weight.data.cpu().numpy()
    beta2 = model.bn2.bias.data.cpu().numpy()
    mean2 = model.bn2.running_mean.cpu().numpy()
    var2 = model.bn2.running_var.cpu().numpy()
    eps2 = model.bn2.eps
    
    # Compute folded parameters
    scale2 = gamma2 / np.sqrt(var2 + eps2)
    bias2 = beta2 - (gamma2 * mean2) / np.sqrt(var2 + eps2)

    # Save parameters
    np.save(f'{output_dir}/bn2_scale.npy', scale2)
    np.save(f'{output_dir}/bn2_bias.npy', bias2)
    np.savetxt(f'{output_dir}/bn2_scale.txt', scale2, fmt='%.6f')
    np.savetxt(f'{output_dir}/bn2_bias.txt', bias2, fmt='%.6f')
    
    print(f"BN2: scale shape={scale2.shape}, bias shape={bias2.shape}")

    return {
    'bn1_scale': scale1, 'bn1_bias': bias1,
    'bn2_scale': scale2, 'bn2_bias': bias2
    }

In [13]:
# Recreate test_loader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Run all exports
print("EXPORTING MODEL FOR FPGA IMPLEMENTATION")

weights = export_binary_weights(model_final, './fpga_weights')
bn_params = export_batchnorm_params(model_final, './fpga_weights')  # Use fixed version!
generate_test_vectors(model_final, test_loader, num_samples=10, output_dir='./fpga_weights')

print("EXPORT COMPLETE")

EXPORTING MODEL FOR FPGA IMPLEMENTATION
FC1: (256, 784), unique values: [-1.  1.]
FC2: (256, 256), unique values: [-1.  1.]
FC3: (10, 256), unique values: [-1.  1.]
BN1: scale shape=(256,), bias shape=(256,)
BN2: scale shape=(256,), bias shape=(256,)


NameError: name 'generate_test_vectors' is not defined

In [13]:
# VERIFICATION OF EXPORTED FILES

# Verify binary weights
fc1 = np.load('./fpga_weights/fc1_weights.npy')
fc2 = np.load('./fpga_weights/fc2_weights.npy')
fc3 = np.load('./fpga_weights/fc3_weights.npy')

print("\nBinary Weight Matrices:")
print(f"  FC1: {fc1.shape} - {fc1.size:,} weights")
print(f"    Values: {np.unique(fc1)}")
print(f"    +1 count: {(fc1 == 1).sum():,}")
print(f"    -1 count: {(fc1 == -1).sum():,}")
print(f"    Balance: {(fc1 == 1).sum() / fc1.size * 100:.1f}% positive")

print(f"\n  FC2: {fc2.shape} - {fc2.size:,} weights")
print(f"    Values: {np.unique(fc2)}")

print(f"\n  FC3: {fc3.shape} - {fc3.size:,} weights")
print(f"    Values: {np.unique(fc3)}")

# Verify BatchNorm
bn1_scale = np.load('./fpga_weights/bn1_scale.npy')
bn1_bias = np.load('./fpga_weights/bn1_bias.npy')
bn2_scale = np.load('./fpga_weights/bn2_scale.npy')
bn2_bias = np.load('./fpga_weights/bn2_bias.npy')

print("\nBatchNorm Parameters:")
print(f"  BN1 scale: shape={bn1_scale.shape}, range=[{bn1_scale.min():.4f}, {bn1_scale.max():.4f}]")
print(f"  BN1 bias:  shape={bn1_bias.shape}, range=[{bn1_bias.min():.4f}, {bn1_bias.max():.4f}]")
print(f"  BN2 scale: shape={bn2_scale.shape}, range=[{bn2_scale.min():.4f}, {bn2_scale.max():.4f}]")
print(f"  BN2 bias:  shape={bn2_bias.shape}, range=[{bn2_bias.min():.4f}, {bn2_bias.max():.4f}]")

# Verify test vectors
test_in = np.load('./fpga_weights/test_inputs.npy')
test_out = np.load('./fpga_weights/test_outputs.npy')
test_labels = np.load('./fpga_weights/test_labels.npy')

print("\nTest Vectors:")
print(f"  Inputs:  shape={test_in.shape}")
print(f"  Outputs: shape={test_out.shape}")
print(f"  Labels:  shape={test_labels.shape}")
print(f"  Predictions vs Labels match: {(test_out.argmax(axis=1) == test_labels).all()}")



Binary Weight Matrices:
  FC1: (256, 784) - 200,704 weights
    Values: [-1.  1.]
    +1 count: 96,137
    -1 count: 104,567
    Balance: 47.9% positive

  FC2: (256, 256) - 65,536 weights
    Values: [-1.  1.]

  FC3: (10, 256) - 2,560 weights
    Values: [-1.  1.]

BatchNorm Parameters:
  BN1 scale: shape=(256,), range=[0.1498, 0.3413]
  BN1 bias:  shape=(256,), range=[-1.7905, 1.6114]
  BN2 scale: shape=(256,), range=[1.1110, 2.1869]
  BN2 bias:  shape=(256,), range=[-3.5600, 4.4523]

Test Vectors:
  Inputs:  shape=(10, 1, 28, 28)
  Outputs: shape=(10, 10)
  Labels:  shape=(10,)
  Predictions vs Labels match: True


In [15]:
"""
Reference:
    PyTorch inference: https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
"""

import torch
import numpy as np
import time


def software_baseline_testbench(model, test_loader, device='cpu'):
    """
    Comprehensive single-pass testbench
    
    Processes all 10,000 MNIST test images exactly once
    Measures timing, accuracy, and throughput
    
    Args:
        model: Trained 1-bit BNN model (TFC_Progressive)
        test_loader: MNIST test DataLoader
        device: 'cpu' or 'cuda'
    
    Returns:
        results: Dictionary with all metrics
    """
    
    print("="*70)
    print("SOFTWARE BASELINE TESTBENCH")
    print("="*70)
    print(f"Device: {device}")
    print(f"Architecture: 784 → 256 → 256 → 10 (1-bit BNN)")
    print(f"Test set: MNIST (10,000 images)")
    print("="*70)
    
    # Set model to evaluation mode
    model.eval()
    model = model.to(device)
    
    # Warm-up: Run a few inferences to stabilize timing
    dummy_input = torch.randn(32, 1, 28, 28).to(device)
    for _ in range(10):
        with torch.no_grad():
            _ = model(dummy_input)
    print("Warm-up complete")

    # TESTBENCH
    print("\nProcessing 10,000 test images...")
    print("-" * 70)
    
    # Tracking variables
    correct = 0
    total = 0
    batch_count = 0
    individual_times = []  # Store time for each batch
    
    # Start total timing
    start_total = time.perf_counter()
    
    # Process all test images
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            # Move to device
            images = images.to(device)
            labels = labels.to(device)
            
            batch_size_current = images.size(0)
            
            # Time this batch
            start_batch = time.perf_counter()
            outputs = model(images)
            end_batch = time.perf_counter()
            
            batch_time = end_batch - start_batch
            individual_times.append(batch_time)
            
            # Get predictions
            _, predicted = outputs.max(1)
            
            # Count correct
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            batch_count += 1
    
    # End total timing
    end_total = time.perf_counter()

    # METRICS
    
    total_time = end_total - start_total
    accuracy = 100. * correct / total
    
    # Timing metrics
    avg_time_per_image_ms = (total_time / total) * 1000
    throughput_images_per_sec = total / total_time
    
    # Batch timing statistics
    individual_times = np.array(individual_times)
    avg_batch_time = np.mean(individual_times) * 1000  # ms
    min_batch_time = np.min(individual_times) * 1000
    max_batch_time = np.max(individual_times) * 1000
    
    # Computational metrics
    ops_fc1 = 784 * 256
    ops_fc2 = 256 * 256
    ops_fc3 = 256 * 10
    total_ops_per_inference = ops_fc1 + ops_fc2 + ops_fc3
    total_ops_executed = total_ops_per_inference * total
    ops_per_second = total_ops_per_inference * throughput_images_per_sec
    gops_per_second = ops_per_second / 1e9
    
    print("\n" + "="*70)
    print("TESTBENCH RESULTS")
    print("="*70)
    
    print("\n1. ACCURACY RESULTS:")
    print(f"   Total images processed:  {total:,}")
    print(f"   Correct predictions:     {correct:,}")
    print(f"   Incorrect predictions:   {total - correct:,}")
    print(f"   Accuracy:                {accuracy:.2f}%")
    
    print("\n2. TIMING RESULTS:")
    print(f"   Total execution time:    {total_time:.4f} seconds")
    print(f"   Average time per image:  {avg_time_per_image_ms:.4f} ms")
    print(f"   Throughput:              {throughput_images_per_sec:.2f} images/second")
    
    print("\n3. BATCH TIMING STATISTICS:")
    print(f"   Total batches:           {batch_count}")
    print(f"   Average batch time:      {avg_batch_time:.4f} ms")
    print(f"   Min batch time:          {min_batch_time:.4f} ms")
    print(f"   Max batch time:          {max_batch_time:.4f} ms")
    
    print("\n4. COMPUTATIONAL ANALYSIS:")
    print(f"   Operations per image:    {total_ops_per_inference:,}")
    print(f"   Total operations:        {total_ops_executed:,}")
    print(f"   Operations per second:   {ops_per_second:.2e}")
    print(f"   Throughput:              {gops_per_second:.4f} GOPS")
      

    
    # Store results
    results = {
        'accuracy': {
            'total_images': total,
            'correct': correct,
            'incorrect': total - correct,
            'accuracy_percent': float(accuracy)
        },
        'timing': {
            'total_time_sec': float(total_time),
            'avg_time_per_image_ms': float(avg_time_per_image_ms),
            'throughput_images_per_sec': float(throughput_images_per_sec)
        },
        'batch_stats': {
            'total_batches': batch_count,
            'avg_batch_time_ms': float(avg_batch_time),
            'min_batch_time_ms': float(min_batch_time),
            'max_batch_time_ms': float(max_batch_time)
        },
        'computational': {
            'ops_per_image': total_ops_per_inference,
            'total_ops_executed': total_ops_executed,
            'ops_per_second': float(ops_per_second),
            'gops_per_second': float(gops_per_second)
        }
    }
    
    return results

In [16]:
results = software_baseline_testbench(model_final, test_loader, device='cpu')

SOFTWARE BASELINE TESTBENCH
Device: cpu
Architecture: 784 → 256 → 256 → 10 (1-bit BNN)
Test set: MNIST (10,000 images)
Warm-up complete

Processing 10,000 test images...
----------------------------------------------------------------------


C:\GitRepos\EECE4632\Project\bnn_env\lib\site-packages\torch\_tensor.py:1679: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/core/TensorImpl.h:1976.)
  return super().rename(names)



TESTBENCH RESULTS

1. ACCURACY RESULTS:
   Total images processed:  10,000
   Correct predictions:     9,850
   Incorrect predictions:   150
   Accuracy:                98.50%

2. TIMING RESULTS:
   Total execution time:    1.2868 seconds
   Average time per image:  0.1287 ms
   Throughput:              7771.12 images/second

3. BATCH TIMING STATISTICS:
   Total batches:           79
   Average batch time:      1.7656 ms
   Min batch time:          1.2355 ms
   Max batch time:          2.5345 ms

4. COMPUTATIONAL ANALYSIS:
   Operations per image:    268,800
   Total operations:        2,688,000,000
   Operations per second:   2.09e+09
   Throughput:              2.0889 GOPS


In [22]:
# First check if CUDA is available
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    
    # Run GPU benchmark
    print("\n" + "="*70)
    print("GPU BENCHMARK")
    print("="*70)
    
    results_gpu = software_baseline_testbench(model_final, test_loader, device='cuda')
    
    # Also run CPU for direct comparison
    print("\n" + "="*70)
    print("CPU BENCHMARK")
    print("="*70)
    
    results_cpu = software_baseline_testbench(model_final, test_loader, device='cpu')
    
    # Print comparison
    print("\n" + "="*70)
    print("CPU vs GPU COMPARISON")
    print("="*70)
    
    cpu_latency = results_cpu['timing']['avg_time_per_image_ms']
    gpu_latency = results_gpu['timing']['avg_time_per_image_ms']
    speedup = cpu_latency / gpu_latency
    
    cpu_throughput = results_cpu['timing']['throughput_images_per_sec']
    gpu_throughput = results_gpu['timing']['throughput_images_per_sec']
    
    print(f"\nLatency (ms/image):")
    print(f"  CPU:  {cpu_latency:.4f} ms")
    print(f"  GPU:  {gpu_latency:.4f} ms")
    print(f"  GPU Speedup: {speedup:.2f}×")
    
    print(f"\nThroughput (images/sec):")
    print(f"  CPU:  {cpu_throughput:.2f} images/sec")
    print(f"  GPU:  {gpu_throughput:.2f} images/sec")
    print(f"  GPU Speedup: {gpu_throughput/cpu_throughput:.2f}×")
    
    print("\n" + "="*70)
    
else:
    print("CUDA not available - running CPU only")
    results_cpu = software_baseline_testbench(model_final, test_loader, device='cpu')

CUDA available: True
CUDA device: NVIDIA GeForce RTX 5070 Ti Laptop GPU

GPU BENCHMARK
SOFTWARE BASELINE TESTBENCH
Device: cuda
Architecture: 784 → 256 → 256 → 10 (1-bit BNN)
Test set: MNIST (10,000 images)
Warm-up complete

Processing 10,000 test images...
----------------------------------------------------------------------

TESTBENCH RESULTS

1. ACCURACY RESULTS:
   Total images processed:  10,000
   Correct predictions:     9,850
   Incorrect predictions:   150
   Accuracy:                98.50%

2. TIMING RESULTS:
   Total execution time:    1.0858 seconds
   Average time per image:  0.1086 ms
   Throughput:              9209.52 images/second

3. BATCH TIMING STATISTICS:
   Total batches:           79
   Average batch time:      1.5538 ms
   Min batch time:          1.2622 ms
   Max batch time:          3.0272 ms

4. COMPUTATIONAL ANALYSIS:
   Operations per image:    268,800
   Total operations:        2,688,000,000
   Operations per second:   2.48e+09
   Throughput:            

In [20]:
baseline_summary = {
    'cpu': {
        'latency_ms': 0.1060,
        'throughput': 9434.86,
        'accuracy': 98.50,
        'gops': 2.54
    },
    'gpu': {
        'latency_ms': 0.1206,
        'throughput': 8292.38,
        'accuracy': 98.50,
        'gops': 2.23,
        'speedup_vs_cpu': 0.88
    }
}

import json
with open('./fpga_weights/baseline_comparison.json', 'w') as f:
    json.dump(baseline_summary, f, indent=2)

In [14]:
def generate_layer_header_simple(layer_num, weights_npy_path, output_path):
    """
    Generate weight header for HLS implementation
    """
    
    weights = np.load(weights_npy_path)
    num_neurons, input_size = weights.shape
    
    with open(output_path, 'w') as f:
        f.write(f"// Auto-generated weights for Layer {layer_num}\n")
        f.write(f"// Shape: {num_neurons} neurons × {input_size} inputs\n")
        f.write(f"// Binary values: -1, +1\n\n")
        
        f.write("#ifndef LAYER{}_WEIGHTS_H\n".format(layer_num))
        f.write("#define LAYER{}_WEIGHTS_H\n\n".format(layer_num))
        
        f.write(f"const int LAYER{layer_num}_WEIGHTS[{num_neurons}][{input_size}] = {{\n")
        
        for i, row in enumerate(weights):
            f.write("    {")
            # Write each weight value
            for j, val in enumerate(row):
                f.write(f"{int(val)}")
                if j < len(row) - 1:
                    f.write(", ")
            f.write("}")
            if i < len(weights) - 1:
                f.write(",\n")
            else:
                f.write("\n")
        
        f.write("};\n\n")
        f.write("#endif\n")
    
    print(f"  Generated {output_path}")
    print(f"  Neurons: {num_neurons}, Inputs: {input_size}")
    print(f"  File size: {os.path.getsize(output_path) / 1024:.1f} KB")


# Usage:
generate_layer_header_simple(1, './fpga_weights/fc1_weights.npy', './fpga_weights/layer1_weights.h')
generate_layer_header_simple(2, './fpga_weights/fc2_weights.npy', './fpga_weights/layer2_weights.h')
generate_layer_header_simple(3, './fpga_weights/fc3_weights.npy', './fpga_weights/layer3_weights.h')

  Generated ./fpga_weights/layer1_weights.h
  Neurons: 256, Inputs: 784
  File size: 692.1 KB
  Generated ./fpga_weights/layer2_weights.h
  Neurons: 256, Inputs: 256
  File size: 227.0 KB
  Generated ./fpga_weights/layer3_weights.h
  Neurons: 10, Inputs: 256
  File size: 9.2 KB
